# Week 1 Exercise (Agentic Course)

### Includes
1. Tool to present sample work
2. RAG pipeline to ingest personal knowledge base

In [ ]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI

In [ ]:
# Install dependencies
# !uv pip install langchain-chroma langchain-huggingface
# !uv pip install scikit-learn

In [ ]:

load_dotenv(override=True)
openai = OpenAI()

MODEL = "gpt-4.1-nano"
db_name = "vector_db"

In [ ]:
### RAG pipeline

# Load in everything in the knowledgebase using LangChain's loaders
folder = "me/"  # the only folder containing .md files
doc_type = os.path.basename(folder)
loader = DirectoryLoader(folder, glob="*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
documents = []

folder_docs = loader.load()
for doc in folder_docs:
    doc.metadata["doc_type"] = doc_type  # optional: label docs with their folder
    documents.append(doc)

# print(f"Loaded {len(documents)} documents")

# Divide into chunks using the RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

# Embeddings & Vector Store
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
retriever = vectorstore.as_retriever()

In [ ]:
# Tools

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

def present_work_sample():
    repo_url = "https://github.com/Andela-AI-Engineering-Bootcamp/Odyssey-Healthy-Food-Assistant"
    return repo_url

In [ ]:
# JSON for tools
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

present_work_sample_json = {
    "name": "present_work_sample",
    "description": "Use this tool to present a work sample or portfolio project to the user",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json},
        {"type": "function", "function": present_work_sample_json}]

In [ ]:

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
reader = PdfReader("me/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("me/Profile_summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Rithwik Mutyala"

In [ ]:
system_prompt_template = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
You are also given a knowledge base of {name}'s work experience, skills, and projects which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. \
If the user is enquiring about your work sample, guide them to the github repository using present_work_sample tool. \
{{context}}"

system_prompt_template += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt_template += f"With this context, please chat with the user, always staying in character as {name}."


In [ ]:
def chat(message, history):
    # RAG: retrieve relevant context
    docs = retriever.invoke(message)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = system_prompt_template.replace("{context}", context)

    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False

    while not done:
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason

        if finish_reason == "tool_calls":
            message_obj = response.choices[0].message
            tool_calls = message_obj.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message_obj)
            messages.extend(results)
        else:
            done = True

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat, type="messages").launch(inbrowser=True)